In [1]:
import torch.nn as nn
import torch
import math

In [13]:
class Attention(nn.Module):
    def __init__(self, dim, num_heads=1, bias=False, attn_dropout=0., proj_dropout=0.):
        super().__init__()
        assert dim % num_heads == 0, "dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.d = math.sqrt(self.head_dim)
        
        #Input dim?
        self.qkv = nn.Linear(dim, dim * 3, bias=bias)
        self.attn_dropout = nn.Dropout(attn_dropout)
        self.proj = nn.Linear(dim, dim)
        self.proj_dropout = nn.Dropout(proj_dropout)
    
    # X has shape [B, D]
    def forward(self, x):
        batch_size, _ = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(batch_size, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(1, 2, 0, 3)
        q, k, v = torch.unbind(qkv, dim=0)

        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn_weights = attn_weights.softmax(dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        out = torch.matmul(attn_weights, v)
        out = out.transpose(0, 1).reshape(batch_size, self.num_heads * self.head_dim)
        out = self.proj(out)
        out = self.proj_dropout(out)
        
        return out

In [14]:
class AttentionTimm(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, attn_drop=0., proj_drop=0.):
        super().__init__()
        assert dim % num_heads == 0, 'dim should be divisible by num_heads'
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x)
        qkv = qkv.reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)   # make torchscript happy (cannot use tensor as tuple)

        attn = (q @ k.transpose(-2, -1)) / math.sqrt(head_dim)
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

In [15]:
num_heads = 4
embed_dim = 256
head_dim = embed_dim // num_heads
batch_size = 128
x = torch.rand(batch_size, embed_dim)

In [16]:
a = AttentionTimm(embed_dim, num_heads)
timm_out = a(x.unsqueeze(0))
timm_out

tensor([[[ 0.0352, -0.1188, -0.2228,  ...,  0.2088,  0.2594, -0.1968],
         [ 0.0349, -0.1190, -0.2229,  ...,  0.2085,  0.2595, -0.1961],
         [ 0.0350, -0.1185, -0.2229,  ...,  0.2087,  0.2594, -0.1960],
         ...,
         [ 0.0349, -0.1188, -0.2228,  ...,  0.2083,  0.2595, -0.1962],
         [ 0.0346, -0.1184, -0.2234,  ...,  0.2085,  0.2596, -0.1966],
         [ 0.0349, -0.1190, -0.2227,  ...,  0.2085,  0.2595, -0.1959]]],
       grad_fn=<ViewBackward0>)

In [17]:
a2 = Attention(embed_dim, num_heads)
a2.qkv = a.qkv
a2.proj = a.proj
out = a2(x)
out

tensor([[ 0.0352, -0.1188, -0.2228,  ...,  0.2088,  0.2594, -0.1968],
        [ 0.0349, -0.1190, -0.2229,  ...,  0.2085,  0.2595, -0.1961],
        [ 0.0350, -0.1185, -0.2229,  ...,  0.2087,  0.2594, -0.1960],
        ...,
        [ 0.0349, -0.1188, -0.2228,  ...,  0.2083,  0.2595, -0.1962],
        [ 0.0346, -0.1184, -0.2234,  ...,  0.2085,  0.2596, -0.1966],
        [ 0.0349, -0.1190, -0.2227,  ...,  0.2085,  0.2595, -0.1959]],
       grad_fn=<AddmmBackward0>)

In [18]:
torch.allclose(out, timm_out)

True

In [8]:
queries = x
keys = x
values = x
true_output, _ = torch.nn.functional.multi_head_attention_forward(
            queries,
            keys,
            values,
            embed_dim_to_check=embed_dim,
            num_heads=num_heads,
            in_proj_weight=a.qkv.weight,
            in_proj_bias=a.qkv.bias,
            bias_k=None,
            bias_v=None,
            add_zero_attn=False,
            dropout_p=0.0,
            out_proj_bias=a.proj.bias,
            out_proj_weight=a.proj.weight,
            training=False,
            key_padding_mask=None,
            need_weights=True,
            attn_mask=None,
            use_separate_proj_weight=False,
            q_proj_weight=None,
            k_proj_weight=None,
            v_proj_weight=None,
            static_k=None,
            static_v=None,
        )
true_output

tensor([[[ 0.0100, -0.0203,  0.5470,  ..., -0.3336, -0.0548, -0.1898]],

        [[-0.4347, -0.2021,  0.4023,  ..., -0.2462, -0.1323, -0.0590]],

        [[-0.1765, -0.3274,  0.6545,  ..., -0.1033, -0.1950, -0.1025]],

        ...,

        [[-0.2087, -0.1356,  0.5584,  ..., -0.1100, -0.1702,  0.0052]],

        [[-0.3040, -0.5702,  0.5443,  ..., -0.2571, -0.1564,  0.1288]],

        [[ 0.0587,  0.0100,  0.5183,  ..., -0.2300,  0.1732, -0.3183]]],
       grad_fn=<UnsafeViewBackward0>)


tensor([[ 0.2603, -0.0173,  0.0362,  ..., -0.2343,  0.0111, -0.1483],
        [ 0.2606, -0.0172,  0.0361,  ..., -0.2345,  0.0108, -0.1480],
        [ 0.2606, -0.0172,  0.0359,  ..., -0.2344,  0.0111, -0.1485],
        ...,
        [ 0.2608, -0.0173,  0.0361,  ..., -0.2345,  0.0109, -0.1483],
        [ 0.2602, -0.0174,  0.0358,  ..., -0.2348,  0.0109, -0.1484],
        [ 0.2598, -0.0174,  0.0366,  ..., -0.2346,  0.0107, -0.1483]],
       grad_fn=<SqueezeBackward1>)

In [9]:
torch.allclose(timm_out.squeeze()[0], true_output[0])

True

In [10]:
torch.allclose(out, true_output)

True